In [ ]:
!python3 -m pip install -r requirements.txt

In [ ]:
import boto3
session = boto3.Session()
print(session.get_credentials().get_frozen_credentials())

In [ ]:
# Send and process a document with Amazon Nova on Amazon Bedrock.

import boto3
import os
from botocore.exceptions import ClientError

here = os.getcwd()
filename = os.path.join(here, 'requirements.txt')

print(f"Using file: {filename}")

# Create a Bedrock Runtime client in the AWS Region you want to use.
client = boto3.client("bedrock-runtime", region_name="us-east-1")

# Set the model ID, e.g. Amazon Nova Lite.
model_id = "amazon.nova-lite-v1:0"

# Load the document
with open(filename, "rb") as file:
    document_bytes = file.read()

# Start a conversation with a user message and the document
conversation = [
    {
        "role": "user",
        "content": [
            {"text": "Briefly describe the package requirements described in this document"},
            {
                "document": {
                    # Available formats: html, md, pdf, doc/docx, xls/xlsx, csv, and txt
                    "format": "txt",
                    "name": "requirements",
                    "source": {"bytes": document_bytes},
                }
            },
        ],
    }
]

try:
    # Send the message to the model, using a basic inference configuration.
    response = client.converse(
        modelId=model_id,
        messages=conversation,
        inferenceConfig={"maxTokens": 500, "temperature": 0.3},
    )

    # Extract and print the response text.
    response_text = response["output"]["message"]["content"][0]["text"]
    print(response_text)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1) 

In [ ]:
from strands import Agent, tool
from strands_tools import calculator, current_time, python_repl
from dotenv import load_dotenv
import os

load_dotenv()


# Define a custom tool as a Python function using the @tool decorator
@tool
def letter_counter(word: str, letter: str) -> int:
    """
    Count occurrences of a specific letter in a word.

    Args:
        word (str): The input word to search in
        letter (str): The specific letter to count

    Returns:
        int: The number of occurrences of the letter in the word
    """
    if not isinstance(word, str) or not isinstance(letter, str):
        return 0

    if len(letter) != 1:
        raise ValueError("The 'letter' parameter must be a single character")

    return word.lower().count(letter.lower())

model_id = "amazon.nova-lite-v1:0"
# model_id = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")

# Create an agent with tools from the strands-tools example tools package
# as well as our custom letter_counter tool
agent = Agent(
    model=model_id,
    tools=[calculator, current_time, python_repl, letter_counter]
)

# Ask the agent a question that uses the available tools
message = """
I have 4 requests:

1. What is the time right now?
2. Calculate 3111696 / 74088
3. Tell me how many letter R's are in the word "strawberry" 🍓
4. Output a script that does what we just spoke about!
Use your python tools to confirm that the script works before outputting it
"""
agent(message)

In [ ]:
 #!python prereqs/knowledge_base.py --mode create

from prereqs.knowledge_base_2 import KnowledgeBasesForAmazonBedrock
kb = KnowledgeBasesForAmazonBedrock(
    model_id="amazon.nova-lite-v1:0",
    knowledge_base_name="restaurant-reservation-system",
    knowledge_base_description="Knowledge base for the restaurant reservation system",
    knowledge_base_id="restaurant-reservation-system-kb",
    knowledge_base_source="
    https://raw.githubusercontent.com/izirhi/restaurant-reservation-system/main/prereqs/knowledge_base.json")
